# 04 - 机械臂与工作台平面的系统级验证

将 aligned depth 反投影并通过手眼矩阵变换到 robot base，使用已知工作台平面检查
`intrinsics + alignment + depth + hand-eye` 的整体 residual。

本 notebook 与 `D:\HIWI\Hand_eye_calibration.ipynb` 使用同一约定：

```text
T_A_B：把 B 坐标系中的点转换到 A 坐标系
p_base = T_base_gripper @ T_gripper_camera @ p_camera
```

这里把参考文件中的 `gripper` 视为当前实验的 `ee`，因此
`T_gripper_camera == T_ee_cam`，`T_base_gripper == T_base_ee`。

**必须手动确认**：手眼标定时使用的 gripper/EE frame 与当前机器人位姿完全相同；
工作台平面或 TCP 接触点正确；每帧 RGB-D 与 `T_base_gripper` 严格同步。

> **运行方式**：从上到下逐个运行。深度数据内部统一使用米（m），统计结果同时显示毫米（mm）。
> 带有 `TODO` 的变量需要按实际实验配置修改。实时采集 cell 需要连接 RealSense D435/D435i。


## Block 1：导入库和参数设置

配置采样网格、文件路径和输出目录。


In [1]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "cam_calibration").is_dir():
            return candidate
    raise RuntimeError("无法定位项目根目录；请从 Xinyu 项目内启动 Jupyter。")

PROJECT_ROOT = find_project_root()
CALIBRATION_ROOT = PROJECT_ROOT / "cam_calibration"
INPUT_DIR = CALIBRATION_ROOT / "inputs"
OUTPUT_ROOT = CALIBRATION_ROOT / "outputs"
SHARED_DIR = OUTPUT_ROOT / "shared"
CONFIG_DIR = CALIBRATION_ROOT / "config"
for directory in (INPUT_DIR, OUTPUT_ROOT, SHARED_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

patch_size = 7
sample_grid = (7, 7)
grid_fraction = (0.25, 0.75, 0.25, 0.75)  # xmin,xmax,ymin,ymax
output_dir = OUTPUT_ROOT / "04_system_validation"
output_dir.mkdir(parents=True, exist_ok=True)

hand_eye_result_path = CONFIG_DIR / "hand_eye_result.json"
intrinsics_path = SHARED_DIR / "camera_intrinsics.json"
robot_poses_path = INPUT_DIR / "robot_poses.csv"
depth_calibration_path = SHARED_DIR / "depth_calibration_linear.json"
rgbd_manifest_path = INPUT_DIR / "rgbd_manifest.csv"
use_depth_correction = True
robot_translation_scale_to_m = 1.0

if patch_size % 2 == 0:
    raise ValueError("patch_size 必须为奇数。")
print("Hand-eye result:", hand_eye_result_path)
print("Camera intrinsics:", intrinsics_path)
print("Robot poses:", robot_poses_path)


Hand-eye result: /home/jingyun/Xinyu/ais_hiwi/cam_calibration/config/hand_eye_result.json
Camera intrinsics: /home/jingyun/Xinyu/ais_hiwi/cam_calibration/outputs/shared/camera_intrinsics.json
Robot poses: /home/jingyun/Xinyu/ais_hiwi/cam_calibration/inputs/robot_poses.csv


## Block 2：加载 depth calibration

如果文件不存在，自动回退为不修正。


In [2]:
if use_depth_correction and depth_calibration_path.exists():
    calibration = json.loads(depth_calibration_path.read_text(encoding="utf-8"))
    a, b = float(calibration["a"]), float(calibration["b"])
    print(f"已加载 D_corr={a:.9f}*D_meas+{b:.9f}")
else:
    a, b = 1.0, 0.0
    use_depth_correction = False
    print("未使用 depth correction。")


未使用 depth correction。


## Block 3：加载手眼标定结果

读取 notebook 03 输出的 `cam_calibration/config/hand_eye_result.json`。
其中 `T_gripper_camera` 满足 `p_gripper = T_gripper_camera @ p_camera`，可直接作为本 notebook 的
`T_ee_cam`。如果手动提供相反方向 `T_camera_gripper`，代码会自动求逆。


In [3]:
def validate_transform(T, name):
    T = np.asarray(T, dtype=float)
    if T.shape != (4, 4) or not np.allclose(T[3], [0, 0, 0, 1], atol=1e-7):
        raise ValueError(f"{name} 不是合法齐次矩阵。")
    R = T[:3, :3]
    if not np.allclose(R.T @ R, np.eye(3), atol=1e-3) or not np.isclose(np.linalg.det(R), 1.0, atol=1e-3):
        raise ValueError(f"{name} 的旋转矩阵不合法。")
    return T

if hand_eye_result_path.exists():
    hand_eye_result = json.loads(hand_eye_result_path.read_text(encoding="utf-8"))
    if hand_eye_result.get("T_gripper_camera") is None:
        raise RuntimeError(
            f"{hand_eye_result_path} 还是模板；请连接机械臂并写入真实 4x4 T_gripper_camera。"
        )
    T_gripper_camera = validate_transform(
        hand_eye_result["T_gripper_camera"], "T_gripper_camera"
    )
    T_ee_cam = T_gripper_camera
    print("已加载手眼方法:", hand_eye_result.get("method", "unknown"))
    print("坐标约定:", hand_eye_result.get(
        "coordinate_convention", "p_gripper = T_gripper_camera @ p_camera"
    ))
else:
    handeye_input_type = "T_gripper_camera"  # 或 "T_camera_gripper"
    T_handeye_input = np.array([  # TODO: 仅在 JSON 不存在时填写
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1],
    ], dtype=float)
    T_gripper_camera = (
        T_handeye_input
        if handeye_input_type == "T_gripper_camera"
        else np.linalg.inv(T_handeye_input)
    )
    T_ee_cam = validate_transform(T_gripper_camera, "T_gripper_camera")
    if np.allclose(T_gripper_camera, np.eye(4)):
        raise FileNotFoundError(
            f"缺少真实手眼结果: {hand_eye_result_path}。"
            "请稍后填写 cam_calibration/config/hand_eye_result.json；禁止使用单位矩阵占位值继续验证。"
        )
    print("未找到 JSON，当前使用已明确填写的手动手眼矩阵。")

print("T_gripper_camera (= T_ee_cam) =\n", T_ee_cam)
print("translation [m]:", T_ee_cam[:3, 3])
if np.allclose(T_ee_cam, np.eye(4)):
    print("警告：当前是单位矩阵占位值，请替换为真实手眼标定结果。")


已加载手眼方法: HORAUD
坐标约定: p_gripper = T_gripper_camera @ p_camera
T_gripper_camera (= T_ee_cam) =
 [[-0.03681367  0.99930495 -0.00586305 -0.04560145]
 [-0.99930482 -0.03677784  0.00610505  0.0375417 ]
 [ 0.00588518  0.00608373  0.99996418  0.05826217]
 [ 0.          0.          0.          1.        ]]
translation [m]: [-0.04560145  0.0375417   0.05826217]


## Block 4：输入或拟合工作台平面

可直接输入 `n_base,d`，或从至少 3 个 TCP 接触点用 SVD 拟合。


In [ ]:
plane_mode = "config"  # "config"、"direct" 或 "fit_points"
n_base_direct = np.array([0.0, 0.0, 1.0])
d_direct = -0.0
tcp_contact_points = np.array([  # TODO: 至少 3 个非共线点，单位 m
    [0.30, -0.20, 0.00],
    [0.50, -0.20, 0.00],
    [0.30,  0.10, 0.00],
    [0.50,  0.10, 0.00],
])

def normalize_plane(n, d):
    norm = np.linalg.norm(n)
    if norm < 1e-12:
        raise ValueError("平面法向量不能为零。")
    return np.asarray(n, float) / norm, float(d) / norm

def fit_plane(points):
    points = np.asarray(points, float)
    if points.ndim != 2 or points.shape[0] < 3 or points.shape[1] != 3:
        raise ValueError("至少需要 3 个三维接触点。")
    center = points.mean(axis=0)
    _, _, vh = np.linalg.svd(points - center)
    n = vh[-1]
    if n[2] < 0: n = -n
    return normalize_plane(n, -n @ center)

plane_config_path = CONFIG_DIR / "table_plane.json"
if plane_mode == "config":
    if not plane_config_path.exists():
        raise FileNotFoundError(
            f"缺少工作台平面配置: {plane_config_path}。"
            "可填写该文件，或把 plane_mode 改为 direct/fit_points。"
        )
    plane_config = json.loads(plane_config_path.read_text(encoding="utf-8"))
    n_base, d = normalize_plane(plane_config["n_base"], plane_config["d"])
elif plane_mode == "direct":
    n_base, d = normalize_plane(n_base_direct, d_direct)
elif plane_mode == "fit_points":
    n_base, d = fit_plane(tcp_contact_points)
    contact_residuals = tcp_contact_points @ n_base + d
    plt.figure(figsize=(7,4)); plt.plot(contact_residuals*1000, "o-")
    plt.axhline(0, color="k", ls="--"); plt.ylabel("Plane residual [mm]")
    plt.xlabel("TCP contact point"); plt.grid(); plt.show()
print(f"Plane: {n_base}^T P + ({d:.9f}) = 0")


## Block 5：加载机器人位姿 `T_base_gripper`

直接读取主流程使用的 `inputs/robot_poses.csv`：
`sample_id,image_path,t00,t01,...,t33`，矩阵按行展开且单位为米。
内部同时提供别名 `T_base_ee`。若数据来自 Franka/libfranka 的列主序 `O_T_EE`，
应先执行 `np.asarray(O_T_EE).reshape(4, 4, order="F")` 再保存。


In [ ]:
POSE_FIELD_NAMES = [f"t{row}{col}" for row in range(4) for col in range(4)]

def load_robot_poses(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"找不到机器人位姿文件: {path}")
    records = []
    if path.suffix.lower() == ".csv":
        table = pd.read_csv(path, encoding="utf-8-sig")
        if all(field in table.columns for field in POSE_FIELD_NAMES):
            for index, row in table.iterrows():
                values = [float(row[field]) for field in POSE_FIELD_NAMES]
                T = np.asarray(values, dtype=float).reshape(4, 4)
                T[:3, 3] *= robot_translation_scale_to_m
                records.append({
                    "sample_id": str(row.get("sample_id", index)),
                    "image_path": str(row.get("image_path", "")),
                    "T_base_gripper": validate_transform(T, f"T_base_gripper[{index}]"),
                })
        else:
            raw = table.to_numpy(dtype=float)
            records = [
                {"sample_id": str(i), "image_path": "",
                 "T_base_gripper": validate_transform(row.reshape(4, 4), f"T_base_gripper[{i}]")}
                for i, row in enumerate(raw)
            ]
    elif path.suffix.lower() == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        matrices = data.get("poses", data) if isinstance(data, dict) else data
        records = [
            {"sample_id": str(i), "image_path": "",
             "T_base_gripper": validate_transform(T, f"T_base_gripper[{i}]")}
            for i, T in enumerate(matrices)
        ]
    else:
        raise ValueError("位姿文件只支持 json/csv。")
    for record in records:
        record["T_base_ee"] = record["T_base_gripper"]
    return records

robot_pose_records = load_robot_poses(robot_poses_path) if robot_poses_path.exists() else []
T_base_ee_list = [record["T_base_gripper"] for record in robot_pose_records]
print("已加载 T_base_gripper 数量:", len(T_base_ee_list))


## Block 6：采集或加载 aligned RGB-D

推荐离线 manifest：列为 `color_path,depth_path`，depth 为 raw uint16 `.npy` 或含 `depth_raw/depth_m` 的 `.npz`。也提供实时采集函数。


In [ ]:
def load_depth_file(path, depth_scale=None):
    path = Path(path)
    if path.suffix.lower() == ".npy":
        arr = np.load(path)
        if depth_scale is None:
            raise ValueError("raw .npy depth 需要 depth_scale。")
        return arr.astype(np.float64) * float(depth_scale)
    data = np.load(path)
    if "depth_m" in data:
        return data["depth_m"].astype(np.float64)
    if "depth_raw" in data and depth_scale is not None:
        return data["depth_raw"].astype(np.float64) * float(depth_scale)
    raise KeyError("NPZ 需要 depth_m，或 depth_raw + depth_scale。")

def capture_aligned_rgbd():
    import pyrealsense2 as rs
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    profile = pipeline.start(config)
    aligner = rs.align(rs.stream.color)
    try:
        for _ in range(30):
            pipeline.wait_for_frames()
        frames = aligner.process(pipeline.wait_for_frames())
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()
        if not color_frame or not depth_frame:
            raise RuntimeError("没有获得 aligned RGB-D 帧。")
        color = np.asanyarray(color_frame.get_data())
        raw = np.asanyarray(depth_frame.get_data())
        scale = float(profile.get_device().first_depth_sensor().get_depth_scale())
        intr = profile.get_stream(rs.stream.color).as_video_stream_profile().get_intrinsics()
        intr_dict = {
            "width": intr.width, "height": intr.height,
            "fx": intr.fx, "fy": intr.fy, "cx": intr.ppx, "cy": intr.ppy,
            "coeffs": list(intr.coeffs),
        }
        return color, raw.astype(np.float64) * scale, intr_dict
    finally:
        pipeline.stop()

rgbd_samples = []
if rgbd_manifest_path.exists():
    manifest = pd.read_csv(rgbd_manifest_path, dtype={"sample_id": str})
    if manifest.empty:
        raise RuntimeError(
            f"{rgbd_manifest_path} 还是空模板。请连接机械臂后填写 RGB-D 样本。"
        )
    if "sample_id" not in manifest.columns:
        raise KeyError("rgbd_manifest.csv 必须包含 sample_id。")
    pose_by_id = {record["sample_id"]: record for record in robot_pose_records}
    missing_pose_ids = sorted(set(manifest["sample_id"]) - set(pose_by_id))
    if missing_pose_ids:
        raise KeyError(f"这些 RGB-D sample_id 缺少机器人位姿: {missing_pose_ids}")
    scale_hint = calibration.get("depth_scale") if "calibration" in globals() else None
    for _, row in manifest.iterrows():
        sample_id = str(row["sample_id"])
        color_path = Path(row["color_path"])
        depth_path = Path(row["depth_path"])
        if not color_path.is_absolute():
            color_path = PROJECT_ROOT / color_path
        if not depth_path.is_absolute():
            depth_path = PROJECT_ROOT / depth_path
        color = cv2.imread(str(color_path)) if str(row.get("color_path", "")).strip() else None
        depth_scale_hint = (
            float(row["depth_scale"])
            if "depth_scale" in row and pd.notna(row["depth_scale"])
            else scale_hint
        )
        depth_m = load_depth_file(depth_path, depth_scale_hint)
        if color is not None and depth_m.shape != color.shape[:2]:
            raise ValueError(f"{sample_id}: depth 未与 color 同尺寸/对齐。")
        rgbd_samples.append({
            "sample_id": sample_id,
            "color": color,
            "depth_m": depth_m,
            "T_base_ee": pose_by_id[sample_id]["T_base_gripper"],
        })
    print("离线 RGB-D 样本:", len(rgbd_samples))
else:
    raise FileNotFoundError(f"缺少 RGB-D manifest: {rgbd_manifest_path}")

## Block 7：在 depth 图中采样网格

每个网格点使用 patch median，过滤 0/NaN。先可视化第一帧确认采样区域确实是桌面。


In [ ]:
def normalize_intrinsics_keys(data):
    data = dict(data)
    if "cx" not in data and "ppx" in data:
        data["cx"] = data["ppx"]
    if "cy" not in data and "ppy" in data:
        data["cy"] = data["ppy"]
    required = ("fx", "fy", "cx", "cy")
    missing = [key for key in required if key not in data]
    if missing:
        raise KeyError(f"相机内参缺少字段: {missing}")
    return data

if intrinsics_path.exists():
    intrinsics = normalize_intrinsics_keys(
        json.loads(intrinsics_path.read_text(encoding="utf-8"))
    )
elif "calibration" in globals() and "camera_intrinsics" in calibration:
    intrinsics = normalize_intrinsics_keys(calibration["camera_intrinsics"])
else:
    raise FileNotFoundError("缺少 camera intrinsics；请先运行参考 hand-eye notebook 或 01/02。")

fx, fy, cx, cy = [float(intrinsics[k]) for k in ("fx", "fy", "cx", "cy")]

def grid_pixels(shape, grid, fractions):
    h, w = shape[:2]; nx, ny = grid
    xmin,xmax,ymin,ymax = fractions
    xs = np.linspace(xmin*w, xmax*w, nx).round().astype(int)
    ys = np.linspace(ymin*h, ymax*h, ny).round().astype(int)
    return [(int(x),int(y)) for y in ys for x in xs]

def patch_median(depth_m, px, py, size):
    half = size//2
    patch = depth_m[max(0,py-half):py+half+1, max(0,px-half):px+half+1]
    valid = patch[np.isfinite(patch) & (patch > 0)]
    return float(np.median(valid)) if valid.size else np.nan

if not rgbd_samples:
    raise RuntimeError("rgbd_samples 为空。请在 Block 6 加载或采集数据。")
pixels = grid_pixels(rgbd_samples[0]["depth_m"].shape, sample_grid, grid_fraction)
if rgbd_samples[0]["color"] is not None:
    vis = rgbd_samples[0]["color"].copy()
    for px,py in pixels: cv2.circle(vis,(px,py),3,(0,255,0),-1)
    plt.figure(figsize=(10,7)); plt.imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB))
    plt.title("Sampling grid: confirm all points are on table"); plt.axis("off"); plt.show()


## Block 8：反投影到 camera 坐标

分别使用原始与线性修正深度生成 `P_cam_meas/P_cam_corr`。


In [ ]:
def deproject(px, py, depth):
    return np.array([(px-cx)/fx*depth, (py-cy)/fy*depth, depth], dtype=float)

point_rows = []
for sample_idx, sample in enumerate(rgbd_samples):
    for px, py in grid_pixels(sample["depth_m"].shape, sample_grid, grid_fraction):
        d_meas = patch_median(sample["depth_m"], px, py, patch_size)
        if not np.isfinite(d_meas) or d_meas <= 0:
            continue
        d_corr = a*d_meas+b if use_depth_correction else d_meas
        point_rows.append({"sample_idx": sample_idx, "px": px, "py": py,
                           "D_meas": d_meas, "D_corr": d_corr,
                           "P_cam_meas": deproject(px,py,d_meas),
                           "P_cam_corr": deproject(px,py,d_corr)})
print("有效采样点:", len(point_rows))


## Block 9：变换到 robot base

应用 `T_base_cam = T_base_ee @ T_ee_cam`。


In [ ]:
def transform_point(T, p):
    return (T @ np.r_[p, 1.0])[:3]

for row in point_rows:
    T_base_ee = rgbd_samples[row["sample_idx"]]["T_base_ee"]
    T_base_cam = T_base_ee @ T_ee_cam
    row["T_base_cam"] = T_base_cam
    row["P_base_before"] = transform_point(T_base_cam, row["P_cam_meas"])
    row["P_base_after"] = transform_point(T_base_cam, row["P_cam_corr"])


## Block 10：计算工作台平面 residual

统计修正前后 residual，并检查其随像素位置和距离的结构性变化。


In [ ]:
def metrics_mm(values):
    x=np.asarray(values)*1000
    return {"mean":x.mean(),"std":x.std(),"MAE":np.mean(np.abs(x)),
            "RMSE":np.sqrt(np.mean(x*x)),"max_abs":np.max(np.abs(x))}
for row in point_rows:
    row["residual_before"] = float(n_base @ row["P_base_before"] + d)
    row["residual_after"] = float(n_base @ row["P_base_after"] + d)
residual_before=np.array([r["residual_before"] for r in point_rows])
residual_after=np.array([r["residual_after"] for r in point_rows])
display(pd.DataFrame([metrics_mm(residual_before),metrics_mm(residual_after)],index=["before","after"]))

pxs=np.array([r["px"] for r in point_rows]); pys=np.array([r["py"] for r in point_rows])
depths=np.array([r["D_meas"] for r in point_rows])
fig,axes=plt.subplots(2,2,figsize=(13,9))
axes[0,0].hist(residual_before*1000,bins=30,alpha=.6,label="before")
axes[0,0].hist(residual_after*1000,bins=30,alpha=.6,label="after"); axes[0,0].legend()
for ax,x,label in [(axes[0,1],pxs,"image x"),(axes[1,0],pys,"image y"),(axes[1,1],depths,"D_meas [m]")]:
    ax.scatter(x,residual_before*1000,s=10,alpha=.5,label="before")
    ax.scatter(x,residual_after*1000,s=10,alpha=.5,label="after")
    ax.axhline(0,color="k",ls="--"); ax.set(xlabel=label,ylabel="Residual [mm]"); ax.legend()
fig.tight_layout(); fig.savefig(output_dir/"plane_residuals.png",dpi=150); plt.show()


## Block 11：由平面几何反推 `D_true`

射线与平面求交；过滤近平行射线，并可独立拟合一组系统级 depth 参数作为诊断。


In [ ]:
valid_geometry_rows=[]
for row in point_rows:
    q=np.array([(row["px"]-cx)/fx,(row["py"]-cy)/fy,1.0])
    R=row["T_base_cam"][:3,:3]; t=row["T_base_cam"][:3,3]
    denominator=float(n_base @ R @ q)
    if abs(denominator)<1e-8:
        row["D_true"]=np.nan
        continue
    D_true=-(float(n_base@t)+d)/denominator
    row["D_true"]=D_true
    if np.isfinite(D_true) and D_true>0:
        valid_geometry_rows.append(row)
geom_meas=np.array([r["D_meas"] for r in valid_geometry_rows])
geom_true=np.array([r["D_true"] for r in valid_geometry_rows])
if len(geom_meas)>=2:
    a_plane,b_plane=np.linalg.lstsq(np.c_[geom_meas,np.ones_like(geom_meas)],geom_true,rcond=None)[0]
    print(f"平面反推拟合: D_corr={a_plane:.9f}*D_meas+{b_plane:.9f}")
print("有效射线-平面交点:",len(valid_geometry_rows))


## Block 12：保存系统验证结果

CSV 保存标量列，NPZ 保存三维点，JSON 保存矩阵、平面与统计。


In [ ]:
save_rows=[]
for r in point_rows:
    save_rows.append({k:r.get(k,np.nan) for k in
                      ["sample_idx","px","py","D_meas","D_corr","D_true","residual_before","residual_after"]})
result_df=pd.DataFrame(save_rows)
result_df.to_csv(output_dir/"system_validation_points.csv",index=False)
np.savez_compressed(output_dir/"system_validation_points.npz",
                    **{c:result_df[c].to_numpy() for c in result_df.columns},
                    P_base_before=np.array([r["P_base_before"] for r in point_rows]),
                    P_base_after=np.array([r["P_base_after"] for r in point_rows]))
summary={
    "T_gripper_camera": T_ee_cam.tolist(),
    "T_ee_cam": T_ee_cam.tolist(),
    "coordinate_convention": "p_base = T_base_gripper @ T_gripper_camera @ p_camera",
    "hand_eye_source": str(hand_eye_result_path),
    "plane":{"n_base":n_base.tolist(),"d":float(d)},
    "depth_correction":{"enabled":use_depth_correction,"a":a,"b":b},
    "metrics_before_mm":metrics_mm(residual_before),
    "metrics_after_mm":metrics_mm(residual_after),
    "camera_intrinsics":intrinsics,
}
(output_dir/"system_validation_summary.json").write_text(
    json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
(SHARED_DIR/"system_geometry.json").write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
print("结果已保存。")


## Block 13：结论提示

RMSE 明显下降说明 depth correction 改善了系统 3D；否则根据 residual 的结构继续定位。


In [ ]:
before=metrics_mm(residual_before)["RMSE"]; after=metrics_mm(residual_after)["RMSE"]
improvement=1-after/before if before>0 else 0
print(f"Plane RMSE: {before:.3f} -> {after:.3f} mm ({improvement:.1%})")
if use_depth_correction and improvement>=0.15:
    print("结论：系统性 depth scale/bias 修正确实改善了系统 3D。")
elif after>5:
    print("结论：残差仍较大，更像 hand-eye、工作台平面、alignment、内参或采样区域的系统误差。")
else:
    print("结论：系统 residual 较小；剩余误差可能主要是局部随机深度噪声。")
